In [ ]:
# 12. Visualization of results
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_venn import venn2

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Peak counts by category
categories = ['Male\nOriginal', 'Female\nOriginal', 'Male\n20kb filtered', 'Female\n20kb filtered', 
              'Male\nMerged', 'Female\nMerged']
counts = [len(peak_data_Male), len(peak_data_Female), len(peak_annotation_Male_20kb), 
          len(peak_annotation_Female_20kb), len(male_merged_df), len(female_merged_df)]
colors = ['steelblue', 'coral', 'steelblue', 'coral', 'steelblue', 'coral']

axes[0, 0].bar(categories, counts, color=colors, alpha=0.7)
axes[0, 0].set_ylabel('Peak Count')
axes[0, 0].set_title('Peak Counts by Processing Step')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Gene coverage
gene_cats = ['Male\nOnly', 'Female\nOnly', 'Shared']
gene_counts = [len(male_unique_genes - female_unique_genes),
               len(female_unique_genes - male_unique_genes),
               len(male_unique_genes & female_unique_genes)]
axes[0, 1].bar(gene_cats, gene_counts, color=['steelblue', 'coral', 'green'], alpha=0.7)
axes[0, 1].set_ylabel('Gene Count')
axes[0, 1].set_title('Gene Coverage by Sex')

# 3. Venn diagram of genes
try:
    venn2([male_unique_genes, female_unique_genes], set_labels=('Male', 'Female'), ax=axes[1, 0])
    axes[1, 0].set_title('Gene Overlap Between Sexes')
except:
    axes[1, 0].text(0.5, 0.5, 'Gene overlap visualization', ha='center', va='center')
    axes[1, 0].set_title('Gene Overlap Between Sexes')

# 4. Public ChIP overlap (if available)
if 'male_vs_public_df' in locals():
    overlap_data = {
        'Male CC': [len(male_merged_df), len(male_vs_public_df)],
        'Female CC': [len(female_merged_df), len(female_vs_public_df)]
    }
    x_pos = range(len(overlap_data))
    widths = 0.35
    
    total_peaks = [v[0] for v in overlap_data.values()]
    overlap_peaks = [v[1] for v in overlap_data.values()]
    
    axes[1, 1].bar(x_pos, total_peaks, widths, label='Total Peaks', alpha=0.7)
    axes[1, 1].bar(x_pos, overlap_peaks, widths, bottom=0, label='Overlap with Public ChIP', alpha=0.7)
    axes[1, 1].set_ylabel('Peak Count')
    axes[1, 1].set_title('Overlap with Public ChIP-seq')
    axes[1, 1].set_xticks(x_pos)
    axes[1, 1].set_xticklabels(overlap_data.keys())
    axes[1, 1].legend()
else:
    axes[1, 1].text(0.5, 0.5, 'Public ChIP comparison not available', ha='center', va='center')

plt.tight_layout()
plt.savefig('Egr1_CC_Comparison_Summary.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Egr1_CC_Comparison_Summary.png', dpi=150, bbox_inches='tight')
print("Summary visualization saved as Egr1_CC_Comparison_Summary.pdf and .png")

In [ ]:
# 12b. Public overlap heatmaps
import seaborn as sns

summary_file = os.path.join(output_dir, 'egr1_public_comparison_summary.txt')
summary_df = pd.read_csv(summary_file, sep='\t')

summary_df['Query'] = summary_df['Dataset'].str.replace('20kb', ' 20kb')
summary_df['Public'] = summary_df['Dataset'].replace({
    'Male20kb_vs_PublicEgr1ChIP': 'Public ChIP',
    'Female20kb_vs_PublicEgr1ChIP': 'Public ChIP',
    'Male20kb_vs_PublicEgr1CUTRUN': 'Public CUT&RUN',
    'Female20kb_vs_PublicEgr1CUTRUN': 'Public CUT&RUN'
})
summary_df['Query'] = summary_df['Query'].str.replace('_vs_.*', '', regex=True)
summary_df['Query'] = summary_df['Query'].replace({'Male 20kb': 'Male 20kb', 'Female 20kb': 'Female 20kb'})

count_matrix = summary_df.pivot(index='Query', columns='Public', values='Overlap_count')
pct_matrix = summary_df.pivot(index='Query', columns='Public', values='Overlap_pct_of_query')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

sns.heatmap(count_matrix, annot=True, fmt='g', cmap='rocket', linewidths=0.5, ax=axes[0])
axes[0].set_title('Overlap Counts')
axes[0].set_ylabel('Query Peak Set')
axes[0].set_xlabel('Public Data Set')

sns.heatmap(pct_matrix, annot=True, fmt='.1f', cmap='viridis', linewidths=0.5, ax=axes[1])
axes[1].set_title('Overlap % of Query Peaks')
axes[1].set_ylabel('')
axes[1].set_xlabel('Public Data Set')

plt.suptitle('Egr1 20kb Peak Overlap with Public Egr1 Data', y=1.02)
plt.savefig(os.path.join(output_dir, 'Egr1_public_data_overlap_heatmaps.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(output_dir, 'Egr1_public_data_overlap_heatmaps.png'), dpi=150, bbox_inches='tight')
print('Saved heatmaps to Egr1_public_data_overlap_heatmaps.pdf and .png')

## Section 12: Visualization of comparison results

In [ ]:
# 11. Export annotated peak tables and summary statistics

# Save full annotated tables with gene information
save_df(peak_annotation_Male_genes, "Male_Egr1CC_peaks_full_withGenes.txt")
save_df(peak_annotation_Female_genes, "Female_Egr1CC_peaks_full_withGenes.txt")

# Create summary statistics
summary_stats = {
    'Dataset': ['Male Egr1 CC', 'Female Egr1 CC', 'Male (20kb filtered)', 'Female (20kb filtered)',
                'Male (merged/extended)', 'Female (merged/extended)', 'Male vs Female Overlap'],
    'Count': [
        len(peak_data_Male),
        len(peak_data_Female),
        len(peak_annotation_Male_20kb),
        len(peak_annotation_Female_20kb),
        len(male_merged_df),
        len(female_merged_df),
        len(overlap_df) if 'overlap_df' in locals() else 0
    ]
}

summary_df = pd.DataFrame(summary_stats)
save_df(summary_df, "Peak_Summary_Statistics.txt")

# Gene coverage summary
gene_summary = {
    'Category': ['Male unique genes', 'Female unique genes', 'Shared genes', 'All genes'],
    'Count': [
        len(male_unique_genes - female_unique_genes),
        len(female_unique_genes - male_unique_genes),
        len(male_unique_genes & female_unique_genes),
        len(combined_unique_genes)
    ]
}

gene_summary_df = pd.DataFrame(gene_summary)
save_df(gene_summary_df, "Gene_Summary_Statistics.txt")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated files:")
print("  - Male_Egr1CC_peaks_full_withGenes.txt")
print("  - Female_Egr1CC_peaks_full_withGenes.txt")
print("  - Male_Egr1CC_peaks_20kbThreshhold_annotated.txt")
print("  - Female_Egr1CC_peaks_20kbThreshhold_annotated.txt")
print("  - Male_Egr1CC_peaks_extended_merged.bed")
print("  - Female_Egr1CC_peaks_extended_merged.bed")
print("  - Male_unique_genes_fromCC_peaks.txt")
print("  - Female_unique_genes_fromCC_peaks.txt")
print("  - Combined_unique_genes_fromCC_peaks.txt")
print("  - Male_vs_Female_peak_overlaps_extended.txt")
print("  - Male_CC_vs_PublicChIP_overlaps.txt")
print("  - Female_CC_vs_PublicChIP_overlaps.txt")
print("  - Peak_Summary_Statistics.txt")
print("  - Gene_Summary_Statistics.txt")

## Section 11: Export summary statistics and final tables

In [ ]:
# 10. Compare with public ChIP-seq data
import os

print("Available public ChIP-seq peak files:")
public_peaks = [f for f in os.listdir(public_data_dir) if f.endswith(('.bed', '.bedGraph'))]
for f in sorted(public_peaks)[:10]:  # Show first 10
    print(f"  {f}")

# Load public peak files
def load_peak_file(filepath):
    """Load peak file with flexible column detection"""
    df = pd.read_csv(filepath, sep='\t', header=None)
    # Keep only first 3 columns (chr, start, end)
    return df[[0, 1, 2]].copy()

# Compare with Yin et al. public ChIP-seq data
public_chip_file = os.path.join(public_data_dir, "Egr1_chipseq_peaks.bed")
if os.path.exists(public_chip_file):
    print(f"\nLoading public ChIP-seq peaks from {public_chip_file}...")
    public_peaks_df = load_peak_file(public_chip_file)
    public_peaks_df.columns = ['chrom', 'start', 'end']
    public_peaks_bed = BedTool.from_dataframe(public_peaks_df)
    print(f"Public ChIP-seq peaks: {len(public_peaks_df)}")
    
    # Overlap Male CC with public ChIP
    male_vs_public = male_merged.intersect(public_peaks_bed, wo=True)
    male_vs_public_df = bed_to_df(male_vs_public)
    if len(male_vs_public_df.columns) >= 7:
        male_vs_public_df.columns = ['cc_chrom', 'cc_start', 'cc_end', 
                                     'chip_chrom', 'chip_start', 'chip_end', 'overlap_bp']
    print(f"Male CC peaks overlapping public ChIP: {len(male_vs_public_df)}")
    print(f"% overlap: {100*len(male_vs_public_df)/len(male_merged_df):.1f}%")
    
    # Overlap Female CC with public ChIP
    female_vs_public = female_merged.intersect(public_peaks_bed, wo=True)
    female_vs_public_df = bed_to_df(female_vs_public)
    if len(female_vs_public_df.columns) >= 7:
        female_vs_public_df.columns = ['cc_chrom', 'cc_start', 'cc_end', 
                                       'chip_chrom', 'chip_start', 'chip_end', 'overlap_bp']
    print(f"Female CC peaks overlapping public ChIP: {len(female_vs_public_df)}")
    print(f"% overlap: {100*len(female_vs_public_df)/len(female_merged_df):.1f}%")
    
    # Save overlap comparisons
    male_vs_public_df.to_csv("Male_CC_vs_PublicChIP_overlaps.txt", sep='\t', index=False, header=True)
    female_vs_public_df.to_csv("Female_CC_vs_PublicChIP_overlaps.txt", sep='\t', index=False, header=True)
else:
    print(f"Warning: Public ChIP file not found at {public_chip_file}")

## Section 10: Compare with public ChIP-seq data

In [ ]:
# 9. Intersect Male vs Female peaks (using extended/merged peaks)
print("Finding overlapping peaks between Male and Female Egr1 CC...")

# Using bedtools intersect
overlap_peaks = male_merged.intersect(female_merged, wo=True)
overlap_df = bed_to_df(overlap_peaks)

print(f"Overlapping peak regions: {len(overlap_df)}")
print(overlap_df.head())

# Rename columns for clarity
if len(overlap_df.columns) >= 6:
    overlap_df.columns = ['male_chrom', 'male_start', 'male_end', 
                          'female_chrom', 'female_start', 'female_end', 'overlap_bp']

# Count peaks with each gene nearby
print(f"\nOverlapping peaks summary:")
print(f"Total overlapping regions: {len(overlap_df)}")
print(f"Total overlap bp: {overlap_df['overlap_bp'].sum()}")

# Get genes at overlap regions
print("\nGenes associated with overlapping peaks...")
overlap_genes_male = []
overlap_genes_female = []

for idx, row in overlap_df.iterrows():
    male_start, male_end = int(row['male_start']), int(row['male_end'])
    female_start, female_end = int(row['female_start']), int(row['female_end'])
    
    # Find genes from male peaks in this region
    male_genes = peak_annotation_Male_genes[
        (peak_annotation_Male_genes['Start'] >= male_start) & 
        (peak_annotation_Male_genes['End'] <= male_end)
    ]['NearestGene'].dropna().unique()
    
    # Find genes from female peaks in this region
    female_genes = peak_annotation_Female_genes[
        (peak_annotation_Female_genes['Start'] >= female_start) & 
        (peak_annotation_Female_genes['End'] <= female_end)
    ]['NearestGene'].dropna().unique()
    
    overlap_genes_male.append(';'.join(male_genes) if len(male_genes) > 0 else '.')
    overlap_genes_female.append(';'.join(female_genes) if len(female_genes) > 0 else '.')

overlap_df['Male_Genes'] = overlap_genes_male
overlap_df['Female_Genes'] = overlap_genes_female

# Save overlap table
overlap_df.to_csv("Male_vs_Female_peak_overlaps_extended.txt", sep='\t', index=False, header=True)
print("\nOverlap annotation saved.")

## Section 9: Intersect male vs female peaks and annotate overlaps

In [ ]:
# 8. Build unique gene lists (nearest OR 2nd nearest, combined and deduplicated)

def build_unique_gene_list(annotation_df):
    """
    Create union of nearest and 2nd nearest genes from annotation.
    Returns set of unique genes, excluding NaN/None values.
    """
    genes = set()
    
    # Add nearest genes
    if 'NearestGene' in annotation_df.columns:
        genes.update(annotation_df['NearestGene'].dropna().unique())
    
    # Add 2nd nearest genes
    if 'SecondNearestGene' in annotation_df.columns:
        genes.update(annotation_df['SecondNearestGene'].dropna().unique())
    
    # Remove empty strings
    genes.discard('')
    genes.discard('.')
    
    return genes

# Get unique genes for each sex
male_unique_genes = build_unique_gene_list(peak_annotation_Male_genes)
female_unique_genes = build_unique_gene_list(peak_annotation_Female_genes)

# Combined unique genes (union)
combined_unique_genes = male_unique_genes | female_unique_genes

print(f"Male unique genes (nearest + 2nd nearest): {len(male_unique_genes)}")
print(f"Female unique genes (nearest + 2nd nearest): {len(female_unique_genes)}")
print(f"Combined unique genes (all): {len(combined_unique_genes)}")
print(f"Genes in both: {len(male_unique_genes & female_unique_genes)}")
print(f"Male-specific: {len(male_unique_genes - female_unique_genes)}")
print(f"Female-specific: {len(female_unique_genes - male_unique_genes)}")

# Save unique gene lists
pd.DataFrame({'Gene': sorted(male_unique_genes)}).to_csv(
    "Male_unique_genes_fromCC_peaks.txt", sep='\t', index=False, header=True)
pd.DataFrame({'Gene': sorted(female_unique_genes)}).to_csv(
    "Female_unique_genes_fromCC_peaks.txt", sep='\t', index=False, header=True)
pd.DataFrame({'Gene': sorted(combined_unique_genes)}).to_csv(
    "Combined_unique_genes_fromCC_peaks.txt", sep='\t', index=False, header=True)

print("\nUnique gene lists saved.")

## Section 8: Build unique gene lists (nearest OR 2nd nearest)

In [ ]:
# 7. Extract nearest and 2nd-nearest genes
# Examine the annotation columns to find gene and distance columns
print("Male annotation columns with Gene/Distance info:")
gene_cols = [col for col in peak_annotation_Male_combined.columns if 'Gene' in col or 'Distance' in col]
print(gene_cols)
print("\nExample row:")
print(peak_annotation_Male_combined[gene_cols].head())

# Function to extract nearest and 2nd nearest genes from annotation
def extract_gene_info(annotation_df):
    """
    Extract nearest and 2nd nearest gene names from annotation dataframe.
    Handles both Gene1/Gene2 and Distance1/Distance2 columns.
    """
    result = annotation_df.copy()
    
    # Identify gene columns
    gene_cols = [col for col in annotation_df.columns if 'Gene' in col]
    dist_cols = [col for col in annotation_df.columns if 'Distance' in col]
    
    # Create NearestGene and SecondNearestGene columns
    result['NearestGene'] = annotation_df[gene_cols[0]].where(
        annotation_df[dist_cols[0]].notna(), 
        None
    ) if len(gene_cols) > 0 and len(dist_cols) > 0 else None
    
    result['SecondNearestGene'] = annotation_df[gene_cols[1]].where(
        annotation_df[dist_cols[1]].notna(), 
        None
    ) if len(gene_cols) > 1 and len(dist_cols) > 1 else None
    
    result['NearestDistance'] = annotation_df[dist_cols[0]] if len(dist_cols) > 0 else None
    result['SecondNearestDistance'] = annotation_df[dist_cols[1]] if len(dist_cols) > 1 else None
    
    return result

peak_annotation_Male_genes = extract_gene_info(peak_annotation_Male_combined)
peak_annotation_Female_genes = extract_gene_info(peak_annotation_Female_combined)

print("\nMale peaks with gene info:")
print(peak_annotation_Male_genes[['Chr', 'Start', 'End', 'NearestGene', 'NearestDistance', 
                                   'SecondNearestGene', 'SecondNearestDistance']].head(10))

## Section 7: Extract nearest and 2nd-nearest genes per peak

In [ ]:
# 6. Extend peaks (±300bp) and merge overlapping peaks
slop_extension = 300  # Extend peaks by 300 bp on each side

print(f"Extending Male peaks by ±{slop_extension} bp...")
male_bed = df_to_bed(peak_data_Male, chr_col='Chr', start_col='Start', end_col='End')
male_slopped = male_bed.slop(b=slop_extension, g='mm10')
male_merged = male_slopped.merge()
male_merged_df = bed_to_df(male_merged)
print(f"Male merged peaks (after extension): {len(male_merged_df)}")
print(male_merged_df.head())

print("\n" + "="*80 + "\n")

print(f"Extending Female peaks by ±{slop_extension} bp...")
female_bed = df_to_bed(peak_data_Female, chr_col='Chr', start_col='Start', end_col='End')
female_slopped = female_bed.slop(b=slop_extension, g='mm10')
female_merged = female_slopped.merge()
female_merged_df = bed_to_df(female_merged)
print(f"Female merged peaks (after extension): {len(female_merged_df)}")
print(female_merged_df.head())

# Save extended/merged peaks
male_merged_df.to_csv("Male_Egr1CC_peaks_extended_merged.bed", sep='\t', index=False, header=False)
female_merged_df.to_csv("Female_Egr1CC_peaks_extended_merged.bed", sep='\t', index=False, header=False)

## Section 6: Extend peaks and merge to capture near-but-not-touching peaks

In [ ]:
# 5. Filter by distance to nearest gene (20 kb threshold)
dist_threshold = 20000

peak_annotation_Male_20kb = peak_annotation_Male_combined[
    peak_annotation_Male_combined['Distance1'].abs() <= dist_threshold
].reset_index(drop=True)

peak_annotation_Female_20kb = peak_annotation_Female_combined[
    peak_annotation_Female_combined['Distance1'].abs() <= dist_threshold
].reset_index(drop=True)

print(f"Male peaks within {dist_threshold} bp of nearest gene: {len(peak_annotation_Male_20kb)}")
print(f"Female peaks within {dist_threshold} bp of nearest gene: {len(peak_annotation_Female_20kb)}")

# Save filtered annotations
save_df(peak_annotation_Male_20kb, f"Male_Egr1CC_peaks_20kbThreshhold_annotated.txt")
save_df(peak_annotation_Female_20kb, f"Female_Egr1CC_peaks_20kbThreshhold_annotated.txt")

## Section 5: Filter by nearest gene distance (20 kb)

In [ ]:
# 4. Annotate peaks with pycallingcards
print("Annotating Male peaks...")
peak_annotation_Male = cc.pp.annotation(peak_data_Male, reference="mm10", bedtools_path=bedtools_path)
peak_annotation_Male_combined = cc.pp.combine_annotation(peak_data_Male, peak_annotation_Male)
print(f"Male peaks annotated: {peak_annotation_Male_combined.shape}")
print(peak_annotation_Male_combined.columns.tolist())

print("\n" + "="*80 + "\n")

print("Annotating Female peaks...")
peak_annotation_Female = cc.pp.annotation(peak_data_Female, reference="mm10", bedtools_path=bedtools_path)
peak_annotation_Female_combined = cc.pp.combine_annotation(peak_data_Female, peak_annotation_Female)
print(f"Female peaks annotated: {peak_annotation_Female_combined.shape}")
print(peak_annotation_Female_combined.columns.tolist())

## Section 4: Annotate peaks with pycallingcards

In [ ]:
# 3. Define helper functions

def df_to_bed(df, name_col=None, chr_col='Chr', start_col='Start', end_col='End'):
    """Convert DataFrame to BedTool"""
    bed_df = df[[chr_col, start_col, end_col]].copy()
    if name_col and name_col in df.columns:
        bed_df = df[[chr_col, start_col, end_col, name_col]].copy()
    return BedTool.from_dataframe(bed_df)

def bed_to_df(bed_tool, chr_col='chrom', start_col='start', end_col='end'):
    """Convert BedTool back to DataFrame"""
    return bed_tool.to_dataframe()

def slop_peaks(df, slop_bp=300, chr_col='Chr', start_col='Start', end_col='End'):
    """
    Extend peaks by slop_bp on both sides using pybedtools slop.
    This captures close but non-overlapping peaks.
    """
    bed_tool = df_to_bed(df, chr_col=chr_col, start_col=start_col, end_col=end_col)
    # Use slop with genome file (mm10)
    slopped = bed_tool.slop(b=slop_bp, g='mm10')
    return bed_to_df(slopped)

def merge_peaks(bed_tool):
    """Merge overlapping peaks"""
    return bed_tool.merge()

def save_df(df, filepath):
    """Save DataFrame to TSV"""
    df.to_csv(filepath, sep='\t', index=False, header=True)
    print(f"Saved to {filepath}")

print("Helper functions defined.")

## Section 3: Define helper functions for peak manipulation

In [ ]:
# 2. Load BED files into DataFrames
# Column names for pycallingcards peak files
peak_columns = ['Chr', 'Start', 'End', 'Center', 'Experiment Insertions', 'Background insertions', 
                'Reference Insertions', 'pvalue Reference', 'pvalue Background', 'Fraction Experiment', 
                'TPH Experiment', 'Fraction background', 'TPH background', 'TPH background subtracted', 
                'pvalue_adj Reference']

# Male peaks file
male_bed_file = "Egr1CC_peak_MaleEgr1_VS_MaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed"
peak_data_Male = pd.read_csv(male_bed_file, sep='\t', header=None)
peak_data_Male.columns = peak_columns
print(f"Male Egr1 CC peaks: {len(peak_data_Male)}")
print(peak_data_Male.head())

print("\n" + "="*80 + "\n")

# Female peaks file
female_bed_file = "Egr1CC_peak_FemaleEgr1_VS_FemaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed"
peak_data_Female = pd.read_csv(female_bed_file, sep='\t', header=None)
peak_data_Female.columns = peak_columns
print(f"Female Egr1 CC peaks: {len(peak_data_Female)}")
print(peak_data_Female.head())

## Section 2: Load BED files into DataFrames

In [ ]:
# 1. Import required libraries and set paths
import pandas as pd
import numpy as np
import os
import pybedtools
from pybedtools import BedTool
import pycallingcards as cc
from matplotlib import pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Define paths
bedtools_path = '/ref/rmlab/software/pycallingcards/bin'
data_dir = '/scratch/rmlab/rmlab_shared3/l.llaci/Egr1_paper/testing_CC_forPaper'
public_data_dir = '/scratch/rmlab/rmlab_shared3/l.llaci/Egr1_paper/public_egr1_chipseq'
output_dir = data_dir  # Save outputs in same directory

# Change to working directory
os.chdir(data_dir)
print(f"Working directory: {os.getcwd()}")
print(f"Bedtools path: {bedtools_path}")

# Egr1 CC Peak Annotation and Comparison
## Extract Nearest and 2nd-Nearest Genes with Peak Extension and Overlap Analysis

This notebook loads your Calling Cards (CUT&RUN) peak files, annotates them with nearest genes, extends peaks to capture near-but-not-touching peaks, and compares male vs female peaks and with public ChIP-seq data.